# Univariate analysis: one column at a time

Run each cell in order with **Shift + Enter**. Each step matches a section of the Note.
The data is the Titanic training set (`data/titanic_train.csv`): 891 passengers, 12 columns.
Every chart is a Plotly chart: hover over it to read exact values, drag to zoom, double-click to reset.

## 1. Load the data

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# read the Titanic passengers into a DataFrame
df = pd.read_csv("data/titanic_train.csv")
df.head()

## 2. Categorical columns

### 2.1 Count plot: how many passengers in each group

In [ ]:
# count the passengers in each group of Survived (0 = died, 1 = survived)
df["Survived"].value_counts()

In [ ]:
# the same counts as a bar chart: this is a count plot
counts = df["Survived"].value_counts().sort_index()
fig = px.bar(x=counts.index.astype(str), y=counts.values, text=counts.values,
             labels={"x": "Survived", "y": "Passengers"}, title="Survived: count plot")
fig.show()

A dropdown to switch between the four categorical columns:

In [ ]:
cols = ["Survived", "Pclass", "Sex", "Embarked"]
fig = go.Figure()
for i, col in enumerate(cols):
    # one bar trace per column; only the first is visible at the start
    vc = df[col].value_counts().sort_index()
    fig.add_trace(go.Bar(x=vc.index.astype(str), y=vc.values, text=vc.values, visible=(i == 0), name=col))
# each button shows one trace and hides the others
buttons = [dict(label=col, method="update",
                args=[{"visible": [j == i for j in range(len(cols))]}, {"title.text": f"{col}: count plot"}])
           for i, col in enumerate(cols)]
fig.update_layout(updatemenus=[dict(buttons=buttons, x=1.0, y=1.15)], title="Survived: count plot",
                  yaxis_title="Passengers", template="simple_white")
fig.show()

A question worth asking from the `Pclass` counts: why did more people travel in first class than in second? The average fare per class gives a hint.

In [ ]:
# average fare paid in each class
df.groupby("Pclass")["Fare"].mean().round(2)

### 2.2 Pie chart: the share of each group in percent

In [ ]:
# value_counts(normalize=True) gives shares instead of counts
(df["Survived"].value_counts(normalize=True) * 100).round(2)

In [ ]:
for col in ["Survived", "Pclass", "Sex"]:
    vc = df[col].value_counts()
    # one pie per column, labelled with the percentage of each group
    fig = px.pie(names=vc.index.astype(str), values=vc.values, title=col)
    fig.update_traces(textinfo="label+percent")
    fig.show()

## 3. Numerical columns

### 3.1 Histogram: how many passengers in each age range

In [ ]:
# 5 bins, as a first rough look
fig = px.histogram(df, x="Age", nbins=5, title="Age: histogram with 5 bins")
fig.update_layout(bargap=0.02, yaxis_title="Passengers", template="simple_white")
fig.show()

Move the slider to change the bin width. Too wide hides the shape; too narrow makes it noisy.

In [ ]:
widths = [20, 16, 10, 5, 2, 1]
fig = go.Figure()
for i, w in enumerate(widths):
    # one histogram per bin width; ages 0 to 80
    fig.add_trace(go.Histogram(x=df["Age"], xbins=dict(start=0, end=80.001, size=w), visible=(w == 5)))
steps = [dict(method="update", label=f"{w} years", args=[{"visible": [j == i for j in range(len(widths))]}])
         for i, w in enumerate(widths)]
fig.update_layout(sliders=[dict(active=widths.index(5), steps=steps, currentvalue=dict(prefix="Bin width: "))],
                  xaxis_title="Age (years)", yaxis_title="Passengers", bargap=0.02, template="simple_white")
fig.show()

### 3.2 Density plot: histogram plus a smooth KDE curve

`sns.distplot` is gone from current seaborn. We draw the same picture: a histogram scaled to density, and a KDE curve from scipy.

In [ ]:
import numpy as np
from scipy.stats import gaussian_kde

# the KDE needs the values without missing ages
ages = df["Age"].dropna()
kde = gaussian_kde(ages)          # fits one smooth curve to the 714 ages
x = np.linspace(0, 80, 400)       # ages at which to draw the curve

fig = go.Figure()
# histnorm="probability density" scales the bars so their total area is 1, like the curve
fig.add_trace(go.Histogram(x=ages, xbins=dict(start=0, end=80.001, size=5),
                           histnorm="probability density", name="histogram", opacity=0.5))
fig.add_trace(go.Scatter(x=x, y=kde(x), name="KDE curve", line=dict(width=3)))
fig.update_layout(title="Age: density histogram and KDE curve", xaxis_title="Age (years)",
                  yaxis_title="Density", bargap=0.02, template="simple_white")
fig.show()

### 3.3 Box plot: five numbers and the outliers

In [ ]:
# hover over the box to read min, Q1, median, Q3, max and the outliers
fig = px.box(df, x="Age", points="outliers", title="Age: box plot")
fig.update_traces(quartilemethod="linear")
fig.show()

In [ ]:
# the numbers behind the box, computed with pandas
q1, median, q3 = df["Age"].quantile([0.25, 0.5, 0.75])
iqr = q3 - q1
print(f"Q1 = {q1}, median = {median}, Q3 = {q3}, IQR = {iqr}")
print(f"lower fence = {q1 - 1.5 * iqr}, upper fence = {q3 + 1.5 * iqr}")
# ages above the upper fence: the outliers
df.loc[df["Age"] > q3 + 1.5 * iqr, "Age"].sort_values().tolist()

In [ ]:
# fares: a squashed box and a long tail of outliers
fig = px.box(df, x="Fare", points="outliers", title="Fare: box plot")
fig.update_traces(quartilemethod="linear")
fig.show()

### 3.4 Summary numbers and skewness

In [ ]:
# smallest, largest and average age
df["Age"].min(), df["Age"].max(), df["Age"].mean()

In [ ]:
# skewness: 0 = symmetric, positive = long tail to the right, negative = long tail to the left
df[["Age", "Fare"]].skew()

## 4. All in one: a histogram with a box plot on top

`marginal="box"` puts both views of one column in a single chart.

In [ ]:
fig = px.histogram(df, x="Fare", nbins=50, marginal="box", title="Fare: histogram and box plot")
fig.update_layout(yaxis_title="Passengers", template="simple_white")
fig.show()